# LeetCode Hot 100 - Day 14

## 今日主题：二叉树的深度、右视图与构造

Day 07 和 Day 08 已经打过二叉树的底子，今天开始做面试里更常见的“进阶树题”：

1. 二叉树的直径：递归返回深度的同时，顺手更新答案。
2. 二叉树的右视图：层序遍历的变形。
3. 将有序数组转换为二叉搜索树：分治建树。
4. 二叉树展开为链表：前序遍历 + 改指针，加练题。

这四题有一个共同点：**先想清楚“函数的返回值代表什么”**，代码自然就写出来了。


## 今天怎么学

1. 第一题是今天的核心，重点理解“一个递归函数同时干两件事”。
2. 第二题只是把 Day 08 的层序遍历改一行。
3. 第三题是分治，重点想清楚“为什么取中间那个数”。
4. 第四题先看思路，明天再写也可以。

最低目标：独立写出二叉树的直径和右视图。


## 今日题单

1. 二叉树的直径（LeetCode 543，简单，必做）
2. 二叉树的右视图（LeetCode 199，中等，必做）
3. 将有序数组转换为二叉搜索树（LeetCode 108，简单，必做）
4. 二叉树展开为链表（LeetCode 114，中等，加练）


## 昨日复习

先用 `day13_practice.ipynb` 重写：

1. 字符串解码：栈里存“外层字符串 + 重复次数”。
2. 柱状图中最大的矩形：单调递增栈 + 末尾哨兵。

口述：对顶堆求中位数，两个堆分别存哪一半？


## 今天的树工具

树题和链表题一样，需要先把工具准备好。这两个函数来自 Day 07 和 Day 08，写法和之前完全一致：

- `build_tree(values)`：按 LeetCode 的层序数组建树，`None` 表示空位置。
- `tree_values(root)`：`build_tree` 的反向工具，把树转回层序列表。前面的 Day 没有这个函数，今天补上它，是为了做完题之后能一眼看出“树变成什么样了”。

注意：`tree_values` 输出的列表末尾会省略多余的 `None`，所以拿它和 LeetCode 的示例输入对比时，形状一致就算对。


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


root = build_tree([1, 2, 3, None, 5])
print("树转回列表：", tree_values(root))
print("空树：", tree_values(build_tree([])))


## 题目 1 做题前先补：一个递归函数同时干两件事

先明确“直径”是什么：树上任意两个节点之间最长路径的**边数**。

这条路径有两个特点：

1. 它一定有一个“最高点”，也就是这条路径上离根最近的那个节点；
2. 对每个节点来说，穿过它的最长路径长度 = 左子树的深度 + 右子树的深度。

所以只需要一个后序遍历（先处理左右，再处理自己）：

- 每个节点都算一次 `左深度 + 右深度`，用它更新全局最大值；
- 但**函数的返回值只能是“自己的深度”**，因为父节点需要的是深度，不是直径。

这就是这类题的关键：**函数返回一个信息（深度），顺路更新另一个信息（答案）。**

问题来了：内层函数怎么把答案传给外面？

下面这个写法最简单，先看一遍：

```python
best = [0]          # 用列表装答案
best[0] = 5         # 内层函数修改列表里的元素
print(best)         # [5]
```

列表是可变对象，函数内部改它的元素，不需要任何额外关键字。（Python 里也可以用 `nonlocal`，效果一样，这里先用列表，更好理解。）


In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def demo_depth(node):
    """带打印的深度计算：看清递归是怎么一层层返回的。"""
    if node is None:
        return 0
    left = demo_depth(node.left)
    right = demo_depth(node.right)
    print("节点", node.val, "的左深度", left, "，右深度", right)
    if left > right:
        return left + 1
    return right + 1


root = TreeNode(1)
root.left = TreeNode(2)
root.right = TreeNode(3)
root.left.left = TreeNode(4)
print("整棵树的深度：", demo_depth(root))


# 题目 1：二叉树的直径

LeetCode 543. Diameter of Binary Tree

## 题目描述（改写版）

给你一棵二叉树的根节点 `root`，返回这棵树的**直径**。

直径的定义是：任意两个节点之间最长路径的**边数**。这条路径不一定经过根节点。

## 输入

- `root`：二叉树根节点，节点个数 1 到 10000。

## 输出

返回一个整数，表示最长路径的边数。

## 示例

示例 1：树是 `[1, 2, 3, 4, 5]`（层序），最长路径是 `4 -> 2 -> 1 -> 3`，共 3 条边，返回 3。

示例 2：树是 `[1, 2]`，返回 1。

示例 3：只有一个根节点，返回 0。

## 易漏细节

- 直径是**边数**，不是节点数。两个节点的路径有 1 条边。
- 最长路径不一定穿过根，所以必须对每个节点都算一遍。
- 返回值（深度）和答案（直径）是两件事，不要混在一起。


## 解法名称

**后序遍历 + 顺带更新（Post-Order Traversal with Global Update）**。

## 暴力思路

对每个节点都算一遍“左深度 + 右深度”，取最大值。这样每个节点都要重新遍历一遍子树，时间 O(n²)。

## 优化思路

一次后序遍历解决。写一个函数 `depth(node)`，它的作用是“返回以 node 为根的子树深度”，同时：

1. 递归拿到 `left` 和 `right`；
2. 用 `left + right` 更新全局最大值（这条路径以 node 为最高点）；
3. 返回 `max(left, right) + 1` 作为自己的深度。

时间 O(n)，每个节点访问一次；空间 O(h)，h 是树高（递归栈）。

## 画图跟踪

以 `[1, 2, 3, 4, 5]` 为例：

| 节点 | 左深度 | 右深度 | 更新答案 | 返回 |
| --- | --- | --- | --- | --- |
| 4 | 0 | 0 | 0 | 1 |
| 5 | 0 | 0 | 0 | 1 |
| 2 | 1 | 1 | 2 | 2 |
| 3 | 0 | 0 | 0 | 1 |
| 1 | 2 | 1 | max(2, 3) = 3 | 3 |

答案 3，出现在节点 1（路径 4 -> 2 -> 1 -> 3）。


## 你来写：二叉树的直径

要求：

- 用后序遍历 + 更新全局最大值写。
- 用列表装答案，或者用 `nonlocal`。
- 写完用 `[1,2,3,4,5]`、`[1,2]`、`[1]` 各跑一遍。

先在心里回答：函数返回的是深度还是直径？为什么？


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


# 题目：二叉树的直径
# 解法：后序遍历 + 顺带更新（Post-Order Traversal with Global Update）
# 输入：二叉树根节点 root，节点个数 1 到 10000。
# 目标：求任意两个节点之间最长路径的边数。
# 输出：返回这个边数（整数）。
# 注意：直径是边数不是节点数；最长路径不一定过根，要对每个节点都算；函数返回深度，答案单独用列表装。


def diameter_of_binary_tree(root):
    # 在这里写你的代码
    pass


print(diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])))


In [ ]:
print(diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])))    # 期望 3
print(diameter_of_binary_tree(build_tree([1, 2])))             # 期望 1
print(diameter_of_binary_tree(build_tree([1])))                # 期望 0
print(diameter_of_binary_tree(build_tree([1, 2, 3, 4, None, None, 5])))   # 期望 4


## 参考答案：二叉树的直径

```python
def diameter_of_binary_tree_answer(root):
    best = [0]

    def depth(node):
        if node is None:
            return 0
        left = depth(node.left)
        right = depth(node.right)
        if left + right > best[0]:
            best[0] = left + right
        if left > right:
            return left + 1
        return right + 1

    depth(root)
    return best[0]
```

面试表达：

我用后序遍历。递归函数返回的是“以当前节点为根的子树深度”，同时在每个节点上，用左子树深度加右子树深度去更新全局的最大直径，因为穿过这个节点的最长路径正好由左右两边的深度拼成。最后返回全局最大值。时间 O(n)，空间是递归栈 O(h)。


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


def diameter_of_binary_tree_answer(root):
    best = [0]

    def depth(node):
        if node is None:
            return 0
        left = depth(node.left)
        right = depth(node.right)
        if left + right > best[0]:
            best[0] = left + right
        if left > right:
            return left + 1
        return right + 1

    depth(root)
    return best[0]


print(diameter_of_binary_tree_answer(build_tree([1, 2, 3, 4, 5])))    # 3
print(diameter_of_binary_tree_answer(build_tree([1, 2])))             # 1
print(diameter_of_binary_tree_answer(build_tree([1])))                # 0


## 题目 2 做题前先补：层序遍历再复习一遍

Day 08 的层序遍历模板，今天只改一个地方。

```python
queue = deque([root])
while queue:
    level_size = len(queue)          # 先固定这一层的节点个数
    for i in range(level_size):      # 只处理这一层
        node = queue.popleft()
        ...                          # 这里处理 node
        if node.left is not None:
            queue.append(node.left)
        if node.right is not None:
            queue.append(node.right)
```

关键点：`level_size = len(queue)` 要在循环开始前固定下来。因为循环过程中孩子会不断入队，如果不固定，就会把下一层的节点也算进来。

今天要用的是 `for i in range(level_size)` 里的 `i`：**当 `i == level_size - 1` 时，这个节点就是这一层最右边的那一个。**


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


root = build_tree([1, 2, 3, None, 5, None, 4])
queue = deque([root])

while queue:
    level_size = len(queue)
    level_values = []
    for i in range(level_size):
        node = queue.popleft()
        level_values.append(node.val)
        if node.left is not None:
            queue.append(node.left)
        if node.right is not None:
            queue.append(node.right)
    print("这一层：", level_values, "，最右边的是：", level_values[-1])


# 题目 2：二叉树的右视图

LeetCode 199. Binary Tree Right Side View

## 题目描述（改写版）

想象你站在二叉树的右侧，从右边看过去，能看到的节点值从上到下排成一列，这就是右视图。

请返回右视图的节点值列表。

## 输入

- `root`：二叉树根节点，节点个数 0 到 100。

## 输出

返回一个列表，从上到下依次是每一层最右边的节点值。空树返回空列表。

## 示例

示例 1：树是 `[1, 2, 3, None, 5, None, 4]`，返回 `[1, 3, 4]`。

示例 2：树是 `[1, None, 3]`，返回 `[1, 3]`。

示例 3：空树，返回 `[]`。

## 易漏细节

- 空树要返回空列表，不能报错。
- 每一层只取最后一个节点，不能只沿着 `right` 一直走（右孩子可能为空，但那一层还有别的节点）。
- `level_size` 必须在循环前固定。


## 解法名称

**层序遍历取每层最后一个（BFS Level Order）**。

## 暴力思路

只沿着 `right` 指针一直走。当某个节点的右孩子为空时就会漏掉左边的节点，所以这个思路是错的（但很多人第一次会这么写，值得记住这个坑）。

## 优化思路

标准层序遍历，每一层结束时把最后一个节点放进结果：

- `level_size = len(queue)` 固定这一层的大小；
- 循环 `i` 从 0 到 `level_size - 1`，当 `i == level_size - 1` 时，把节点值加入结果。

时间 O(n)，每个节点进出队列一次；空间 O(w)，w 是树的最大宽度。

如果面试官要求用 DFS，也可以写“优先访问右孩子，记录每个深度第一次见到的节点”，效果一样。


## 你来写：二叉树的右视图

要求：

- 用层序遍历写，注意 `level_size` 要在内层循环前固定。
- 空树返回 `[]`。
- 写完用示例的三组数据各跑一遍。

先在心里回答：为什么不能只沿着 right 指针走？


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


# 题目：二叉树的右视图
# 解法：层序遍历取每层最后一个（BFS Level Order）
# 输入：二叉树根节点 root，节点个数 0 到 100。
# 目标：从上到下取出每一层最右边的节点值。
# 输出：返回这些值组成的列表；空树返回 []。
# 注意：每层开始前固定 level_size；只取每层最后一个；不能只沿着 right 指针走。


def right_side_view(root):
    # 在这里写你的代码
    pass


print(right_side_view(build_tree([1, 2, 3, None, 5, None, 4])))


In [ ]:
print(right_side_view(build_tree([1, 2, 3, None, 5, None, 4])))   # 期望 [1, 3, 4]
print(right_side_view(build_tree([1, None, 3])))                 # 期望 [1, 3]
print(right_side_view(build_tree([])))                           # 期望 []
print(right_side_view(build_tree([1])))                          # 期望 [1]
print(right_side_view(build_tree([1, 2, None, 3])))              # 期望 [1, 2, 3]


## 参考答案：二叉树的右视图

```python
from collections import deque


def right_side_view_answer(root):
    if root is None:
        return []

    result = []
    queue = deque([root])
    while queue:
        level_size = len(queue)
        for i in range(level_size):
            node = queue.popleft()
            if i == level_size - 1:
                result.append(node.val)
            if node.left is not None:
                queue.append(node.left)
            if node.right is not None:
                queue.append(node.right)

    return result
```

面试表达：

我用层序遍历。每轮开始前先记下当前队列的长度，也就是这一层的节点个数；然后只弹出这么多个节点。当处理到这一层的最后一个节点时，把它加入结果，因为它就是从右边看能看到的那个。时间 O(n)，空间是队列的最大宽度。


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


def right_side_view_answer(root):
    if root is None:
        return []

    result = []
    queue = deque([root])
    while queue:
        level_size = len(queue)
        for i in range(level_size):
            node = queue.popleft()
            if i == level_size - 1:
                result.append(node.val)
            if node.left is not None:
                queue.append(node.left)
            if node.right is not None:
                queue.append(node.right)

    return result


print(right_side_view_answer(build_tree([1, 2, 3, None, 5, None, 4])))   # [1, 3, 4]
print(right_side_view_answer(build_tree([1, None, 3])))                 # [1, 3]
print(right_side_view_answer(build_tree([])))                           # []


## 题目 3 做题前先补：二叉搜索树和“平衡”

**二叉搜索树（BST）**的规则：对任意节点，左子树里所有值都小于它，右子树里所有值都大于它。

把 BST 按中序遍历（左 -> 自己 -> 右）走一遍，得到的一定是**升序序列**。反过来，给你一个升序数组，你也能造出一棵 BST。

问题是这样造出来的树可能很偏，比如 `[1, 2, 3, 4, 5]` 一直往右挂，就变成一条链表，查找退化成 O(n)。

**平衡**的意思是左右子树高度差不太大。取数组的**中间元素**当根，左边一半造左子树，右边一半造右子树，自然就平衡了。

这个思路叫**分治**：把大问题切成两个同类型的小问题。


In [ ]:
# 手工看一次分治的切法
nums = [-10, -3, 0, 5, 9]

mid = len(nums) // 2
print("中间下标：", mid, "，根节点是：", nums[mid])
print("左半段：", nums[:mid])
print("右半段：", nums[mid + 1:])

# 再切一次左半段
left_part = nums[:mid]
mid2 = len(left_part) // 2
print("左半段的中间是：", left_part[mid2])


# 题目 3：将有序数组转换为二叉搜索树

LeetCode 108. Convert Sorted Array to Binary Search Tree

## 题目描述（改写版）

给你一个**严格升序**排列的整数数组 `nums`，请把它转换成一棵**高度平衡**的二叉搜索树。

高度平衡的意思是：每个节点的左右子树高度差不超过 1。

如果有多种答案，返回任意一种都可以。

## 输入

- `nums`：严格升序数组，长度 1 到 10000。

## 输出

返回构造出来的二叉搜索树的根节点。

## 示例

示例 1：`nums = [-10, -3, 0, 5, 9]`，一种答案是根节点 0，左子树是 `-10 -> -3` 系，右子树是 `5 -> 9` 系。

示例 2：`nums = [1, 3]`，返回根为 3、左孩子为 1，或者根为 1、右孩子为 3，都算对。

## 易漏细节

- 必须取中间元素当根，才能保证平衡。
- 空数组要返回 `None`，这是递归的终止条件。
- 左半段是 `nums[:mid]`，右半段是 `nums[mid + 1:]`，不要把中间元素重复放进去。


## 解法名称

**分治 + 取中点建树（Divide and Conquer）**。

## 暴力思路

按顺序一个个插入，用普通 BST 插入法。如果数组是升序的，插入结果会变成一条向右的链，高度不平衡，不符合题目要求。

## 优化思路

分治三步：

1. 取数组中间位置 `mid = len(nums) // 2`，用 `nums[mid]` 建根节点；
2. 递归用左半段 `nums[:mid]` 建左子树；
3. 递归用右半段 `nums[mid + 1:]` 建右子树。

因为每次都从中间切，左右两半的元素个数最多差 1，所以树自然是平衡的。

时间 O(n)，每个元素建一个节点。空间上，切片会额外复制数组，总共是 O(n log n)；如果想省掉切片，可以改成传“左右下标”而不是传子数组。

## 画图跟踪

`nums = [-10, -3, 0, 5, 9]`：

```text
mid = 2，根是 0
左边 [-10, -3] -> mid = 1，根是 -3，左孩子 -10
右边 [5, 9]     -> mid = 1，根是 9，左孩子 5

        0
      /   \
    -3     9
   /      /
 -10     5
```


## 你来写：将有序数组转换为二叉搜索树

要求：

- 用分治，每次取中间元素。
- 空数组返回 `None`。
- 写完用一个升序数组建树，再用中序遍历确认结果是从小到大的。

先在心里回答：为什么取中间元素就能保证平衡？


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


# 题目：将有序数组转换为二叉搜索树
# 解法：分治 + 取中点建树（Divide and Conquer）
# 输入：严格升序的整数数组 nums，长度 1 到 10000。
# 目标：把它转换成一棵高度平衡的二叉搜索树。
# 输出：返回树的根节点。
# 注意：每次取中间元素当根；左半段用 nums[:mid]，右半段用 nums[mid + 1:]；空数组返回 None。


def sorted_array_to_bst(nums):
    # 在这里写你的代码
    pass


root = sorted_array_to_bst([-10, -3, 0, 5, 9])
print(tree_values(root))


In [ ]:
root = sorted_array_to_bst([-10, -3, 0, 5, 9])
print(tree_values(root))          # 期望 [0, -3, 9, -10, None, 5]
root2 = sorted_array_to_bst([1, 3])
print(tree_values(root2))         # 期望 [3, 1] 或 [1, None, 3]
print(tree_values(sorted_array_to_bst([])))     # 期望 []
print(tree_values(sorted_array_to_bst([5])))    # 期望 [5]


## 参考答案：将有序数组转换为二叉搜索树

```python
def sorted_array_to_bst_answer(nums):
    if len(nums) == 0:
        return None
    mid = len(nums) // 2
    root = TreeNode(nums[mid])
    root.left = sorted_array_to_bst_answer(nums[:mid])
    root.right = sorted_array_to_bst_answer(nums[mid + 1:])
    return root
```

面试表达：

我用分治。每次取数组的中间元素作为根节点，这样左右两边的元素个数最多差一个，树的高度就是平衡的。然后递归用左半段构造左子树、右半段构造右子树。时间 O(n)，每个元素建一个节点。如果要求空间更优，可以改成传左右下标，避免切片复制数组。


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


def sorted_array_to_bst_answer(nums):
    if len(nums) == 0:
        return None
    mid = len(nums) // 2
    root = TreeNode(nums[mid])
    root.left = sorted_array_to_bst_answer(nums[:mid])
    root.right = sorted_array_to_bst_answer(nums[mid + 1:])
    return root


def inorder_list(root):
    """中序遍历，用来检查是不是有序的（也就是合法 BST）。"""
    result = []
    if root is None:
        return result
    result.extend(inorder_list(root.left))
    result.append(root.val)
    result.extend(inorder_list(root.right))
    return result


root = sorted_array_to_bst_answer([-10, -3, 0, 5, 9])
print(tree_values(root))
print("中序遍历（应该升序）：", inorder_list(root))


## 题目 4 做题前先补：前序遍历的顺序

要展开成链表，先得知道“展开后应该是什么顺序”。

题目给的例子：树是 `[1, 2, 5, 3, 4, None, 6]`，展开后是 `1 -> 2 -> 3 -> 4 -> 5 -> 6`。

这个顺序正好是**前序遍历**（自己 -> 左 -> 右）的顺序。所以最直白的做法是：

1. 前序遍历，把所有节点按顺序装进列表；
2. 按列表顺序，把每个节点的 `left` 置空、`right` 指向下一个节点。

不过文件里没有现成的前序遍历函数，这里补一个 `preorder_nodes(root)`，它和 Day 07 的中序遍历只差“访问自己的位置不同”。


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


def preorder_nodes(root):
    """前序遍历，按 自己 -> 左 -> 右 的顺序把节点装进列表。"""
    result = []
    if root is None:
        return result
    result.append(root)
    result.extend(preorder_nodes(root.left))
    result.extend(preorder_nodes(root.right))
    return result


root = build_tree([1, 2, 5, 3, 4, None, 6])
nodes = preorder_nodes(root)
values = []
for node in nodes:
    values.append(node.val)
print("前序顺序：", values)


# 题目 4：二叉树展开为链表

LeetCode 114. Flatten Binary Tree to Linked List

## 题目描述（改写版）

给你一棵二叉树的根节点 `root`，请把它**原地**展开成一条“只有右孩子的链表”：

- 展开后的顺序和前序遍历一致；
- 每个节点的 `left` 都必须是 `None`；
- 每个节点的 `right` 指向下一个节点；
- 不返回任何东西，直接修改原来的节点。

## 输入

- `root`：二叉树根节点，节点个数 0 到 2000。

## 输出

不需要返回。函数执行完后，原树变成右链。

## 示例

示例 1：树是 `[1, 2, 5, 3, 4, None, 6]`，展开后是 `1 -> 2 -> 3 -> 4 -> 5 -> 6`（全部在 right 上）。

示例 2：空树，什么都不做。

示例 3：只有一个根节点，保持原样。

## 易漏细节

- 是原地修改，不要返回新树。
- 每个节点都要把 `left` 清空，只留 `right`。
- 顺序必须是前序，不是层序。


## 解法名称

**前序遍历 + 重接指针（Preorder Collect and Rewire）**。

## 暴力思路

直接用递归按前序顺序改指针，最麻烦的地方是“右孩子会被覆盖”，改之前必须先把右子树保存下来。容易写错。

## 优化思路

先收集、再重接，思路最清楚：

1. 前序遍历整棵树，把节点按顺序放进列表；
2. 从第一个节点开始，依次让 `当前节点.left = None`、`当前节点.right = 下一个节点`。

时间 O(n)，空间 O(n)（列表 + 递归栈）。

如果面试官要求额外空间 O(1)，可以答“迭代找前驱”的写法：对每个节点，如果它有左孩子，就找到左子树最右边的节点，把当前节点的右子树接到那个节点的右边，再把左孩子挪到右边并清空，然后继续往右走。下面也给了这段代码，先掌握第一种。


## 你来写：二叉树展开为链表

要求：

- 用“前序收集 + 重接”写。
- 注意函数不需要返回值，直接改节点。
- 写完用 `[1,2,5,3,4,None,6]` 跑一遍，从根开始沿着 `right` 走，打印所有值。

先在心里回答：为什么不能一边前序遍历一边直接改指针？


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


# 题目：二叉树展开为链表
# 解法：前序遍历 + 重接指针（Preorder Collect and Rewire）
# 输入：二叉树根节点 root，节点个数 0 到 2000。
# 目标：原地把树展开成只有右孩子的链，顺序与前序遍历一致。
# 输出：不需要返回，直接修改原节点。
# 注意：每个节点的 left 要置为 None；顺序是前序；先收集节点再重接，避免覆盖右孩子。


def flatten(root):
    # 在这里写你的代码
    pass


root = build_tree([1, 2, 5, 3, 4, None, 6])
flatten(root)
current = root
values = []
while current is not None:
    values.append(current.val)
    current = current.right
print(values)


In [ ]:
root = build_tree([1, 2, 5, 3, 4, None, 6])
flatten(root)
current = root
values = []
while current is not None:
    values.append(current.val)
    current = current.right
print(values)                    # 期望 [1, 2, 3, 4, 5, 6]

root2 = build_tree([])
flatten(root2)
print(tree_values(root2))        # 期望 []

root3 = build_tree([1])
flatten(root3)
print(root3.val, root3.left, root3.right)    # 期望 1 None None


## 参考答案：二叉树展开为链表

```python
def flatten_answer(root):
    def preorder(node):
        result = []
        if node is None:
            return result
        result.append(node)
        result.extend(preorder(node.left))
        result.extend(preorder(node.right))
        return result

    nodes = preorder(root)
    for i in range(len(nodes) - 1):
        nodes[i].left = None
        nodes[i].right = nodes[i + 1]
```

面试表达：

我先做一次前序遍历，把节点按访问顺序收进列表，因为展开后的顺序就是前序顺序。然后遍历这个列表，把每个节点的 `left` 置空、`right` 指向下一个节点。最后一个节点的 `right` 本来就是空的，不用处理。时间 O(n)，空间 O(n)。

如果要求额外空间 O(1)，我可以用迭代法：对每个节点，如果它有左孩子，就找到左子树最右边的节点，把当前节点的右子树接到它后面，然后把左孩子移到右边并清空左指针，再继续处理下一个右节点。


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


def flatten_answer(root):
    def preorder(node):
        result = []
        if node is None:
            return result
        result.append(node)
        result.extend(preorder(node.left))
        result.extend(preorder(node.right))
        return result

    nodes = preorder(root)
    for i in range(len(nodes) - 1):
        nodes[i].left = None
        nodes[i].right = nodes[i + 1]


root = build_tree([1, 2, 5, 3, 4, None, 6])
flatten_answer(root)
values = []
current = root
while current is not None:
    values.append(current.val)
    current = current.right
print(values)      # [1, 2, 3, 4, 5, 6]


## 第二层答案：O(1) 额外空间的迭代法

面试官追问“能不能不用列表”时用这一段。

思路：对每个节点，如果它有左孩子，就找到左子树中**最右边的节点**（前序遍历里，左子树最后一个被访问的节点），把当前节点的右子树接到它后面，然后把左孩子挪到右边、清空左指针：

```python
def flatten_second(root):
    current = root
    while current is not None:
        if current.left is not None:
            rightmost = current.left
            while rightmost.right is not None:
                rightmost = rightmost.right
            rightmost.right = current.right
            current.right = current.left
            current.left = None
        current = current.right
```

时间 O(n)，额外空间 O(1)。


# 今日小结

今天四个模式：

1. **一函数两职责**：递归函数返回深度，顺路更新全局答案（列表装答案，或 `nonlocal`）。
2. **层序遍历变形**：固定 `level_size`，取每层最后一个就是右视图。
3. **分治建树**：取中点当根，左右递归，自然平衡。
4. **前序收集 + 重接指针**：需要改指针顺序时，先收集再改最不容易错。

一句话总结：**写递归题先问自己“这个函数返回什么”**。


## 今日复盘区

- 二叉树的直径里，递归函数返回的是深度还是直径？
- 层序遍历里，为什么 `level_size` 必须在循环前固定？
- 为什么取中点建树就能保证平衡？
- 展开为链表为什么是前序顺序？
- 今天哪几道题能不看答案写出来？

完成情况记录：

- 独立写出：
- 卡住的题：
- 明天重写：
- 完成日期：
